# Statistical analysis: are the reliability differences real?

This notebook tests the patterns the dashboards show, using the exported marts and a random
flight-level sample (`data/analysis/flights_sample.parquet`, created by `python -m pipeline.export`).

**Read before interpreting.** With hundreds of thousands of flights, almost any difference is
"statistically significant". So every test below also reports an **effect size** (how big the
difference is) and a confidence interval. Judge findings by effect size and practical impact,
not by p-values alone. Everything here is observational: these tests show association, not cause.

Run top to bottom: *Kernel -> Restart & Run All*.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
MARTS, ANALYSIS = ROOT / "data" / "marts", ROOT / "data" / "analysis"

def load(folder, name):
    parquet = folder / f"{name}.parquet"
    frame = pd.read_parquet(parquet) if parquet.exists() else pd.read_csv(folder / f"{name}.csv")
    for col in frame.columns:  # CSV exports from psql write booleans as t/f
        if pd.api.types.is_string_dtype(frame[col]) and set(frame[col].dropna().unique()) <= {"t", "f"}:
            frame[col] = frame[col].map({"t": True, "f": False})
    return frame

manifest = pd.read_json(MARTS / "manifest.json", typ="series")
if manifest.get("synthetic_data"):
    print("WARNING: SYNTHETIC TEST DATA - results below are meaningless. Load real BTS data first.")

carrier_monthly = load(MARTS, "mart_carrier_monthly")
sample = load(ANALYSIS, "flights_sample")
completed = sample[sample["is_completed"].astype(bool)].copy()
print(f"sample: {len(sample):,} flights ({len(completed):,} completed), "
      f"months {manifest.get('first_month')}-{manifest.get('latest_month')}")

## 1. Is on-time performance independent of the carrier? (chi-square test)

Uses the **full** data (all flights, from `mart_carrier_monthly`), not the sample.
Effect size: **Cramer's V** (0 = no association, ~0.1 small, ~0.3 medium, ~0.5 large).

In [ ]:
by_carrier = carrier_monthly.groupby("carrier_code")[["on_time_flights", "scheduled_flights"]].sum()
table = np.column_stack([by_carrier["on_time_flights"],
                         by_carrier["scheduled_flights"] - by_carrier["on_time_flights"]])
chi2, p_value, dof, _ = stats.chi2_contingency(table)
n = table.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

summary = (by_carrier.assign(on_time_rate=lambda d: d.on_time_flights / d.scheduled_flights)
           .sort_values("on_time_rate"))
print(f"chi-square = {chi2:,.1f}, dof = {dof}, p = {p_value:.2e}, Cramer's V = {cramers_v:.3f}")
summary.style.format({"on_time_rate": "{:.1%}", "on_time_flights": "{:,}", "scheduled_flights": "{:,}"})

## 2. Do evening departures arrive later than morning departures? (Welch's t-test)

Welch's version does not assume equal variances (evening delays vary much more).
Effect size: **Cohen's d** (~0.2 small, ~0.5 medium, ~0.8 large). Delay minutes are heavily
right-skewed, so a Mann-Whitney test is shown as a robustness check.

In [ ]:
morning = completed.loc[completed.time_of_day == "morning", "arr_delay_min"].dropna()
evening = completed.loc[completed.time_of_day == "evening", "arr_delay_min"].dropna()

t_stat, p_t = stats.ttest_ind(evening, morning, equal_var=False)
pooled_sd = np.sqrt((morning.var(ddof=1) + evening.var(ddof=1)) / 2)
cohens_d = (evening.mean() - morning.mean()) / pooled_sd
diff = evening.mean() - morning.mean()
se = np.sqrt(morning.var(ddof=1) / len(morning) + evening.var(ddof=1) / len(evening))
u_stat, p_u = stats.mannwhitneyu(evening, morning, alternative="two-sided")

print(f"morning mean {morning.mean():.1f} min (n={len(morning):,}) | evening mean {evening.mean():.1f} min (n={len(evening):,})")
print(f"difference {diff:.1f} min, 95% CI [{diff - 1.96 * se:.1f}, {diff + 1.96 * se:.1f}]")
print(f"Welch t = {t_stat:.1f}, p = {p_t:.2e}, Cohen's d = {cohens_d:.3f}")
print(f"Mann-Whitney U p = {p_u:.2e} (median {morning.median():.0f} vs {evening.median():.0f} min)")

## 3. Does a late inbound aircraft predict a late flight? (cascade / propagation)

Compares the share of flights arriving 15+ minutes late when the **same aircraft's previous
leg** arrived 15+ late versus on time. Effect size: **risk ratio** with a 95% CI.

In [ ]:
legs = completed.dropna(subset=["prev_leg_arr_delay_min"]).copy()
legs["prev_late"] = legs["prev_leg_arr_delay_min"] >= 15
legs["late"] = legs["is_arr_delayed_15"].astype(bool)

a = legs.loc[legs.prev_late, "late"]; b = legs.loc[~legs.prev_late, "late"]
risk_late, risk_ok = a.mean(), b.mean()
rr = risk_late / risk_ok
se_log_rr = np.sqrt((1 - risk_late) / a.sum() + (1 - risk_ok) / b.sum())
ci = np.exp(np.log(rr) + np.array([-1.96, 1.96]) * se_log_rr)
print(f"P(late | previous leg late) = {risk_late:.1%} (n={len(a):,})")
print(f"P(late | previous leg on time) = {risk_ok:.1%} (n={len(b):,})")
print(f"risk ratio = {rr:.2f}, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
legs.groupby(pd.cut(legs.prev_leg_arr_delay_min, [-np.inf, 0, 15, 30, 60, 120, np.inf]),
             observed=True)["late"].agg(["mean", "size"]).rename(columns={"mean": "share_late"})

## 4. What predicts a 15+ minute arrival delay? (logistic regression)

One model with several factors at once, so each odds ratio is "holding the others fixed".
Odds ratio > 1 = higher odds of delay than the reference group. Reference groups: the first
category alphabetically (shown in the variable name).

In [ ]:
import statsmodels.formula.api as smf

model_data = completed.dropna(subset=["is_arr_delayed_15"]).copy()
model_data["delayed"] = model_data["is_arr_delayed_15"].astype(int)
model_data["hub"] = model_data["origin_is_hub"].astype(int)
model_data["month"] = model_data["month"].astype(str)

model = smf.logit(
    "delayed ~ C(time_of_day, Treatment('morning')) + C(carrier_code) + C(month)"
    " + C(distance_band) + hub",
    data=model_data,
).fit(disp=False)

odds = pd.DataFrame({
    "odds_ratio": np.exp(model.params),
    "ci_low": np.exp(model.conf_int()[0]),
    "ci_high": np.exp(model.conf_int()[1]),
    "p_value": model.pvalues,
}).drop(index="Intercept")
print(f"n = {int(model.nobs):,}, pseudo R-squared = {model.prsquared:.3f}")
odds.sort_values("odds_ratio", ascending=False).round(3)

## 5. Findings (fill in from YOUR results)

Write 3-5 findings here once real data is loaded. For each: the number, its effect size,
and what it does and does not show. Template:

- **Finding:** Evening departures averaged X minutes more arrival delay than morning ones
  (95% CI A-B; Cohen's d = D, a small/medium effect).
- **What it means for operations:** ...
- **Limits:** observational data; time of day is correlated with aircraft rotation position,
  so part of the evening effect may be propagated delay (see section 3).

Possible confounders to discuss: weather days, airport congestion, route length, aircraft
rotation position, carrier network structure, seasonality.